# pip & import

In [ ]:
!pip install -q langchain langchain-community langchain-classic langchain-text-splitters \
    langchain-huggingface langchain-chroma \
    transformers sentence-transformers accelerate \
    pymupdf beautifulsoup4 \
    faiss-cpu rank_bm25 flashrank

In [ ]:
# 표준 라이브러리
import re
import logging
from typing import List
from collections import defaultdict

# 외부 라이브러리
import bs4
import torch
import pandas as pd
from transformers import AutoTokenizer
from google.colab import files

# LangChain Core
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 문서 로드 & 분할
from langchain_community.document_loaders import PyMuPDFLoader, WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 임베딩 & LLM (HuggingFace)
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline, ChatHuggingFace

# 벡터스토어
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS

# 검색기
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import ContextualCompressionRetriever, EnsembleRetriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

# 압축기 & 리랭커
from langchain_classic.retrievers.document_compressors import LLMChainExtractor, CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.document_compressors import FlashrankRerank

# 2-6-2. Contextual Compression


### Document load & Chunking

In [ ]:
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

loader = PyMuPDFLoader(file_name)
data = loader.load()

Saving 디지털 정부혁신 추진계획.pdf to 디지털 정부혁신 추진계획.pdf


splitter: 문서를 청크로 자르는 도구

tokenizer: 글자를 토큰 단위로 세는 도구
- splitter가 청크 하나가 1000토큰 넘지 않게 잘라야 할 때, 토큰 수를 세어주는 역할
- 없이 splitter 쓰면, 글자수 기준으로 자름


In [ ]:
tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-m3')
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=1000,
    chunk_overlap=200,
)

documents = text_splitter.split_documents(data)
print(len(documents))

### Embedding & VectorStore

In [ ]:
# embedding 모델 정의
embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True},
)

# 같은 세션에서 다른 파일로 다시 실행할 때 이전 문서가 섞이지 않도록 기존 컬렉션 삭제
try:
    vectorstore.delete_collection()
except NameError:
    pass

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name='rag_study',
)
print(vectorstore._collection.count())  # 저장된 청크 수

### LLM define

In [ ]:
llm = ChatHuggingFace(llm=HuggingFacePipeline.from_model_id(
    model_id='Qwen/Qwen2.5-3B-Instruct',
    task='text-generation',
    device=0,  # GPU 사용
    pipeline_kwargs={'max_new_tokens': 500, 'do_sample': False, 'return_full_text': False},
))

### base retriever

쿼리와 유사한 청크를 벡터DB에서 찾아옴

In [ ]:
question = '디지털 정부 혁신 추진 계획 중 제일 우선적인 계획을 알려줘.'

In [ ]:
base_retriever = vectorstore.as_retriever(
                                search_type='mmr',
                                search_kwargs={'k':7, 'fetch_k': 20})

base_docs = base_retriever.invoke(question)
print(len(base_docs))

7


### compressor

base retriever가 가져온 문서 중 관련 없는 문서 / 관련 없는 문장을 잘라냄 (LLM 사용)

In [ ]:
compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=base_retriever
)

compressed_docs = compression_retriever.invoke(question)
print(len(compressed_docs))

### compressor 이후 결과 비교

비교 함수 정의

In [ ]:
def words(text):
    return set(text.split())

# 압축 문서마다 단어가 가장 많이 겹치는 원본 문서와 짝짓기
match = {}
for c in compressed_docs:
    overlap = [len(words(c.page_content) & words(b.page_content)) for b in base_docs]
    match[overlap.index(max(overlap))] = c.page_content

rows = []
for i, b in enumerate(base_docs):
    comp = match.get(i)
    rows.append({
        '#': i + 1,
        '원본': b.page_content,
        '압축 결과': comp if comp else '❌ 제거됨',
        '원본 글자수': len(b.page_content),
        '압축 글자수': len(comp) if comp else 0,
    })

In [ ]:
df = pd.DataFrame(rows)
print(f"원본 {len(base_docs)}개 → 압축 {len(compressed_docs)}개 | "
      f"글자수 {df['원본 글자수'].sum():,} → {df['압축 글자수'].sum():,}")

원본 7개 → 압축 1개 | 글자수 5,308 → 286


문서 시각화

In [ ]:
import re
from IPython.display import display

def clean(text):
    """줄바꿈·연속 공백을 한 칸 띄어쓰기로 정리 (보기용)"""
    return re.sub(r'\s+', ' ', text).strip()

view = df.copy()
view['원본'] = view['원본'].apply(clean)
view['압축 결과'] = view['압축 결과'].apply(clean)

def highlight_removed(row):
    """제거된 문서 행은 회색으로 표시"""
    color = 'color: #999999' if row['압축 결과'] == '❌ 제거됨' else ''
    return [color] * len(row)

display(
    view.style
        .apply(highlight_removed, axis=1)
        .set_properties(**{'white-space': 'normal', 'text-align': 'left',
                           'vertical-align': 'top', 'font-size': '13px'})
        .set_properties(subset=['원본'], **{'min-width': '450px'})
        .set_properties(subset=['압축 결과'], **{'min-width': '350px', 'font-weight': 'bold'})
        .set_properties(subset=['#', '원본 글자수', '압축 글자수'], **{'text-align': 'center'})
        .hide(axis='index')
)

#,원본,압축 결과,원본 글자수,압축 글자수
1,순 서 . Ⅰ 개요······························································· 1 디지털 정부혁신 추진계획 . Ⅱ ······················· 2 우선 추진과제 1. ································································ 2 선제적 통합적 대국민 서비스 혁신 · ① 공공부문 마이데이터 활성화 ② 시민참여를 위한 플랫폼 고도화 ③ ④ 현장중심 협업을 지원하는 스마트 업무환경 구현 클라우드와 디지털서비스 이용 활성화 ⑤ 개방형 데이터 서비스 생태계 구축 · ⑥ 중장기 범정부 디지털 전환 로드맵 수립 2. ················ 4 추진체계 및 일정 . Ⅲ ······································· 4 붙임 디지털 정부혁신 우선 추진과제상세 [ ] ( ) ··········· 8,❌ 제거됨,523,0
2,"Ⅱ디지털 정부혁신 추진계획 . ‣ 비전 디지털로 여는 좋은 세상 ( ) 부제 대한민국이 먼저 갑니다 : . ※ ‣ 추진원칙 ( ) 최종 이용자의 관점에서 ① ②공공서비스 수준 향상을 목표로 혁신 친화적인 방식으로 국민과 함께 ③ ④ 우선 추진과제 1 선제적 통합적 대국민 서비스 혁신 · ○ 맞춤형 안내 ( ) 각 기관에서 제공하는 급부적 서비스를 분류 화하고 DB , ‧ 한 번에 안내추천신청결과확인까지 가능한 ‧ ‧ ‧ 통합서비스 환경 구현 * 스마트폰 스피커 등 다양한 기기에서 인공지능 기반의 채팅음성대화 지원 PC, , AI ‧ ○ 생애주기 서비스 ( ) 출산결혼사망 등 · · 생애주기상주요 이벤트와 관련된 서비스를 한 번에 안내받고 신청할 있는 원스톱 패키지 확대 행복출산안심상속 등 종 년 개 이상든든임신온종일돌봄 등 * , 2 ‘22 10 ( , ) → ○ 선제적 서비스 ( ) 사회적 취약계층의 사전동의를 받아 맞춤형 복지 서비스를 선제적으로 지원하는 가칭복지멤버십 도입 ○ 혁신적 서비스 ( ) 프로세스 혁신대국민서비스 향상 등 기존 한계를 , 뛰어넘는 디지털 기반의 공공서비스 혁신사업 적극 발굴·추진 블록체인 기반 부동산거래 플랫폼 인공지능 출입국시스템 등 구축 * (‘20~), (’19~) ○ 서비스표준 ( ) 정부 웹사이트를 쉽고 편리하게 이용할 수 있도록 통합로그인 체계 및 디지털 서비스표준 마련 * 공공 웹사이트 플러그인 완전제거통합 간편로그인 활성화웹사이트 표준화 등 , , UI·UX 2 공공부문 마이데이터 활성화 ○ 자기정보 활용 ( ) 국민이 요청하면 보유기관의 동의 없이도본인 정보를 민원처리에 활용할 수 있도록 행정정보공동이용제도 개선 현행 근거법령 보유기관 동의 * ( ) + ① ② ⇒ 개선 민원인 동의 ( ) ○ 자기정보 다운로드 ( ) 자기정보를 데이터 형태로 다운로드하고 안전 하게 유통위변조 방지유통이력 확인 ( , )할 수 있는 마이데이터 포털구축 * 현행 증명서 제출 ( ) ⇒ 향후 필요 최소한의 데이터 선별다운로드제공 ( ) ‧ ‧",❌ 제거됨,1106,0
3,디지털 정부혁신 추진계획 2019. 10. 29. 관계부처 합동,❌ 제거됨,42,0
4,"참고3 디지털 정부혁신 기대효과 ◈ 동 방안을 성공적으로 추진하는 경우, 정부 디지털 서비스에서 세계 최고 선도국으로 재도약 가능 년대 중반 2020 ⇨ 전자정부 평가 위 UN 1 기대 □ 대국민 ( ) 공공서비스에 대한 이용자 경험을 획기적으로 개선 ‘ ○ 년까지 국민이 22 편리하게 맞춤형으로 공공서비스를 추천안내 · 받고 신청결과확인 · 까지 할 수 있는 기반을 조성하여 이용자 편익 제고 ※ 자신만의 전자지갑사서함을 통해 각종 증명서를 전자적으로 보완제출하고여권만료 / · , · 취업훈련건강검진 등을 사전에 안내고지받고실시간으로 신고신청 가능 · · , · 사회적 취약계층에 대한 선제적 서비스 제공으로 사각지대 예방 - ‘ ○ 년까지 22 종이증명서를 절반 이하로 감축하여 조원 이상 3 절감 ○ 국민에게 친숙한 민간서비스를 통해 공공서비스를 제공하는 등 불편한 정부 온라인 서비스 환경을 글로벌 표준에 맞도록 개선 □ 정부내 ( ) 국민과 현장의 요구에 민감하게 반응할 수 있는 정부로 혁신 대국민 서비스 향상을 위해 ○ 데이터를 막힘없이 연계활용 · 하고, 서비스 수요에 따라 정보시스템의 자원을 유연한 활용이 가능 ○ 사무실 중심 민원처리에서 벗어나 국민의 목소리에 민감하게 반응하고, 안전환경복지 등 · · 증대되는 현장행정 수요에 능동적으로 대처 □ 산업계 ( ) 개방형 생태계를 기반으로 산업 SW 의 글로벌 도약기대 ○ 오픈소스중심의 디지털정부 생태계와 공공시장 수요를 바탕으로 첨단 디지털 산업의 혁신 가속화와글로벌 도약 을 위한 전기 마련 ※ 오픈 를 통한 데이터 및 공공서비스 제공마이데이터 활성화오픈소스 클라우드 API , , 플랫폼 등을 중심으로 국내 산업 혁신 생태계가 크게 활성화될 것으로 기대 SW 장기적으로 정부의 디지털 전환이 적극 추진되어 스마트시티 등 ○ 도시행정 전반의 혁신으로 파급되는 경우 막대한 경제효과 예상 ※ 국내 공공도시분야 디지털 혁신으로 1 ‧ 6~ 년까지 조원 파급효과 30 105 , 10~ 만명 일자리 증가 30 맥킨지 (‘18. )",❌ 제거됨,1074,0
5,"- 범정부 차원의 생애주기 패키지 발굴 프로세스 혁신 시스템 → → 연계구축 ‧ 을 위한 추진체계 강화협업정원정부혁신평가포상 등 활용 ( , , ) 생애주기 패키지 분야 < 예시 ( ) > : 시행중인 서비스, 시행 준비 중인 서비스 : □ 3 사각지대 예방을 위한 선제적 서비스 제공 년말 시행 (21 ) 복지부, 각부처 지자체 · 사회적 취약계층을 대상으로 ○ 사전동의를 받아 주기적 소득재산 ‧ 인적정보 조사 ‧ 와 이력관리를 통해 맞춤형 서비스를선제적으로 * 지원하는 가칭 ( )복지멤버십도입 월 복지부 기발표 (’19.4 ) * 사회보장 지원을 희망하는 개인가구를 대상으로 수급가능성이 높은 사업을 먼저 찾아 안내 ‧ - 핵심 성공요소인 주민가족관계소득금융부동산공공요금 등 정보 및 · · · · · 시스템 연계를 범정부적으로 뒷받침 년 사회보장급여법 개정 (‘20 ) □ 4 기존 한계를 뛰어넘는 공공서비스 혁신사업 추진계속 ( ) 행안 과기정통부 · , 각부처 ○ 디지털 관점에서 △기존 프로세스를 혁신하거나, △공통 플랫폼으로 활용 가능하거나 기타 , △대국민 서비스를 획기적으로 향상시킬 수 있는 디지털 혁신사업 적극 발굴추진 · (디지털 정부혁신 중점사업선정지원 · ) 중점사업예시 ( ) 현행 개선 블록체인 기반 부동산거래 플랫폼 국토부 ( ) * 년 20 년 ISP, 23 구축 완료 부동산 증명서 통합 종 (18 1 ) → 에도 종이문서 유통 여전연간수수료 ( 억 1,300 ), 실거래가 조작등 시장 교란행위 포착 곤란 블록체인으로 관공서법원은행 · · · 전자계약참여자 간 거래정보를 공유문서감축이상거래탐지 ( · )하고, 계약 대출 등기 → → 자동실행 인공지능 식별추적 시스템 ‧ 법무부과기정통부 ( ) ‧ 년 경쟁적 * 19 22 ∼ 개발 공공은 출입국서비스 개선을 위한 출입자 식별과 이상행동 탐지 알고리즘 필요 민간은 / AI 학습데이터 확보 곤란 경쟁적 기술개발 년 (19 22 ) ∼ 로 인식률 99.9%의 식별 알고리즘 등 확보→대기시간 단축, 범죄 율 이상 감소 기대 20%","디지털 정부 혁신 추진 계획 중 제일 우선적인 계획을 알려면, 먼저 생애주기 패키지 발굴과 프로세스 혁신, 연계구축을 통해 범정부 차원의 협업과 평가, 포상 등을 강화하는 방안이 있습니다. 생애주기 패키지 분야에서는 예를 들어, 사회적 취약계층을 대상으로 주기적 소득재산 및 인적 정보 조사와 이력관리를 통해 맞춤형 서비스를 제공하는 가칭 '복지멤버십' 도입이 있습니다. 또한, 디지털 관점에서 기존 프로세스를 혁신하거나 공통 플랫폼으로 활용할 수 있는 디지털 혁신사업을 적극 발굴하고 추천하는 것도 중요합니다.",1086,286
6,참고4 주요 서비스별 변화 모습 선제적 맞춤형서비스 □ 생애주기 원스톱서비스 □ 디지털 고지 활성화 □,❌ 제거됨,59,0

In [ ]:
ㅁ

#2-6-5. reranker
: Cross-Encoder Reranker vs FlashRank

### Document Load

In [3]:
documents = [
    Document(page_content="LangChain은 LLM 애플리케이션 개발 프레임워크입니다."),
    Document(page_content="RAG는 검색 증강 생성의 약자입니다."),
    Document(page_content="Reranker는 검색 결과를 재정렬합니다."),
    Document(page_content="Vector Store는 임베딩을 저장합니다."),
    Document(page_content="BM25는 키워드 기반 검색 알고리즘입니다."),]

### Embedding & VectorStore

In [ ]:
vectorstore = Chroma.from_documents(documents, HuggingFaceEmbeddings())

### Base Retriever

1. Dense Retriever

In [5]:
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

2. BM25 Retriever

In [6]:
bm25_retriever = BM25Retriever.from_documents(documents, k=5)

3. Ensemble Retriever

In [7]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5])

### Reranker

4-1. Cross-Encoder Reranker 추가

In [ ]:
cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
reranker_CE = CrossEncoderReranker(model=cross_encoder, top_n=3)

4-2. FlashRank 추가

In [9]:
reranker_FR= FlashrankRerank(
    model="ms-marco-MultiBERT-L-12",  # 다국어(한국어 포함) 지원 모델
    top_n=3)

ms-marco-MultiBERT-L-12.zip: 100%|██████████| 98.7M/98.7M [00:01<00:00, 82.6MiB/s]


5. 최종 Retriever

: Ensemble + Reranker

Cross-Encoder

In [10]:
final_retriever_CE = ContextualCompressionRetriever(
    base_compressor=reranker_CE,
    base_retriever=ensemble_retriever)

FlashRank

In [11]:
final_retriever_FR = ContextualCompressionRetriever(
    base_compressor=reranker_FR,
    base_retriever=ensemble_retriever)

### 검색 실행 비교

In [15]:
query = "LangChain이란 무엇인가요?"
results_ensemble = ensemble_retriever.invoke(query)[:3]

for i, doc in enumerate(results_ensemble):
    print(f"{i+1}. {doc.page_content}")

1. LangChain은 LLM 애플리케이션 개발 프레임워크입니다.
2. BM25는 키워드 기반 검색 알고리즘입니다.
3. RAG는 검색 증강 생성의 약자입니다.


In [13]:
query = "LangChain이란 무엇인가요?"
results_CE = final_retriever_CE.invoke(query)

for i, doc in enumerate(results_CE):
    print(f"{i+1}. {doc.page_content}")

1. LangChain은 LLM 애플리케이션 개발 프레임워크입니다.
2. BM25는 키워드 기반 검색 알고리즘입니다.
3. Reranker는 검색 결과를 재정렬합니다.


In [14]:
query = "LangChain이란 무엇인가요?"
results_FR = final_retriever_FR.invoke(query)

for i, doc in enumerate(results_FR):
    print(f"{i+1}. {doc.page_content}")

1. LangChain은 LLM 애플리케이션 개발 프레임워크입니다.
2. Vector Store는 임베딩을 저장합니다.
3. BM25는 키워드 기반 검색 알고리즘입니다.


In [ ]:
ㅁ

# 2-6-4. RAG-Fusion
- MultiQueryRetriever
- Rag-Fusion

### document load

In [ ]:
# 기술 뉴스 기사 샘플
documents = [
    Document(
        page_content="OpenAI가 GPT-5를 발표했습니다. 새로운 모델은 이전 버전보다 추론 능력이 크게 향상되었으며, 멀티모달 기능이 강화되었습니다.",
        metadata={"source": "tech_news", "date": "2024-01-15", "topic": "AI"}
    ),
    Document(
        page_content="인공지능 기술의 발전으로 자연어 처리 분야가 빠르게 성장하고 있습니다. 특히 대형 언어 모델(LLM)의 성능 향상이 눈에 띕니다.",
        metadata={"source": "tech_analysis", "date": "2024-01-10", "topic": "AI"}
    ),
    Document(
        page_content="최신 AI 모델들은 코드 생성, 문서 요약, 번역 등 다양한 작업에서 인간 수준의 성능을 보이고 있습니다.",
        metadata={"source": "tech_review", "date": "2024-01-12", "topic": "AI"}
    ),
    Document(
        page_content="GPT 시리즈의 발전 과정을 살펴보면, GPT-3에서 GPT-4로 넘어오면서 논리적 추론과 맥락 이해 능력이 크게 개선되었습니다.",
        metadata={"source": "tech_history", "date": "2024-01-08", "topic": "AI"}
    ),
    Document(
        page_content="기업들이 AI 도입을 가속화하고 있습니다. 특히 고객 서비스, 데이터 분석, 콘텐츠 생성 분야에서 AI 활용이 두드러집니다.",
        metadata={"source": "business_news", "date": "2024-01-14", "topic": "Business"}
    ),
    Document(
        page_content="AI 규제에 대한 논의가 활발합니다. EU는 AI Act를 통해 고위험 AI 시스템에 대한 규제 프레임워크를 마련했습니다.",
        metadata={"source": "policy_news", "date": "2024-01-13", "topic": "Policy"}
    ),
]


### Embedding & VectorStore

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cuda'},   # CPU면 'cpu'
    encode_kwargs={'normalize_embeddings': True},
)
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

### llm 정의 (위에 실행한 셀과 동일)

In [ ]:
llm = ChatHuggingFace(llm=HuggingFacePipeline.from_model_id(
    model_id='Qwen/Qwen2.5-3B-Instruct',
    task='text-generation',
    device=0,  # GPU 사용 (CPU면 -1)
    pipeline_kwargs={'max_new_tokens': 500, 'do_sample': False, 'return_full_text': False},
))

### 다중 쿼리 생성 프롬프트

In [ ]:
query_gen_prompt = ChatPromptTemplate.from_template("""
당신은 문서 검색용 쿼리를 만드는 도우미입니다.
원본 질문에 답하는 데 필요한 문서를 찾기 위한 검색 쿼리 3개를 만드세요.

규칙:
- 원본 질문을 세부 주제로 나눠, 각 쿼리가 서로 다른 측면을 찾도록 하세요.
- 원본 질문에 없는 새로운 요구(논문, 연구, 통계 등)를 추가하지 마세요.
- 문장보다 핵심 키워드 중심의 짧은 검색어로 쓰세요.
- 번호, 기호, 설명 없이 한 줄에 하나씩만 출력하세요.

예시)
원본 질문: 벡터 데이터베이스의 장단점은?
검색 쿼리:
벡터 데이터베이스 장점 유사도 검색 속도
벡터 데이터베이스 단점 비용 한계
벡터 데이터베이스와 관계형 데이터베이스 비교

원본 질문: {question}

검색 쿼리들 (한 줄에 하나씩):
""")

## MultiQueryRetriever

In [ ]:
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=llm,
    prompt=query_gen_prompt,     # 추가: RAG-Fusion과 같은 프롬프트
    include_original=True,
)

### 검색

In [ ]:
logging.basicConfig()
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)

question = "최신 GPT 모델의 성능은 어떤가요?"
docs = multi_query_retriever.invoke(question)

print(f"검색된 문서 수: {len(docs)}")
for i, doc in enumerate(docs, 1):
    print(f"\n[{i}] {doc.page_content[:80]}...")

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['최신 GPT 모델 성능 평가', 'GPT 모델 사용 사례 분석', 'GPT 모델 잠재력 미래 전망']


검색된 문서 수: 5

[1] OpenAI가 GPT-5를 발표했습니다. 새로운 모델은 이전 버전보다 추론 능력이 크게 향상되었으며, 멀티모달 기능이 강화되었습니다....

[2] GPT 시리즈의 발전 과정을 살펴보면, GPT-3에서 GPT-4로 넘어오면서 논리적 추론과 맥락 이해 능력이 크게 개선되었습니다....

[3] 최신 AI 모델들은 코드 생성, 문서 요약, 번역 등 다양한 작업에서 인간 수준의 성능을 보이고 있습니다....

[4] 기업들이 AI 도입을 가속화하고 있습니다. 특히 고객 서비스, 데이터 분석, 콘텐츠 생성 분야에서 AI 활용이 두드러집니다....

[5] 인공지능 기술의 발전으로 자연어 처리 분야가 빠르게 성장하고 있습니다. 특히 대형 언어 모델(LLM)의 성능 향상이 눈에 띕니다....


## Rag-Fusion

### 쿼리 생성 체인

In [ ]:
query_gen_chain = query_gen_prompt | llm | StrOutputParser()

### RRF 함수

In [ ]:
def reciprocal_rank_fusion(
    results_list: List[List[Document]],
    k: int = 60
) -> List[Document]:
    """
    여러 검색 결과 리스트를 RRF로 병합합니다.

    Args:
        results_list: 각 쿼리별 검색 결과 리스트들
        k: 순위 완화 상수 (기본값 60)

    Returns:
        RRF 점수로 정렬된 문서 리스트
    """
    # 문서별 누적 점수 저장 딕셔너리
    fused_scores = defaultdict(float) # 처음 보는 문서는 0점에서 시작
    doc_map = {}  # page_content -> Document 매핑

    for results in results_list:
        for rank, doc in enumerate(results, start=1):
            doc_key = doc.page_content
            fused_scores[doc_key] += 1 / (k + rank)
            doc_map[doc_key] = doc

    # 점수 기준 내림차순 정렬
    sorted_docs = sorted(
        fused_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    # Document 객체로 변환하여 반환
    return [doc_map[content] for content, score in sorted_docs]

### RAG-Fusion 검색 함수
: 앞의 두 셀 조합해 전체 과정 실행

In [ ]:
def rag_fusion_retrieve(question: str, top_k: int = 5) -> List[Document]:
    """RAG-Fusion 검색을 수행합니다."""

    # 1. 다중 쿼리 생성
    queries_text = query_gen_chain.invoke({"question": question})
    queries = [re.sub(r'^\s*(\d+[\.\)]|[-•*])\s*', '', q).strip()
               for q in queries_text.strip().split('\n')]
    queries = [q for q in queries if q and q != question][:3]

    # 원본 질문도 포함
    all_queries = [question] + queries

    print(f"생성된 쿼리 ({len(all_queries)}개):")
    for i, q in enumerate(all_queries):
        print(f"  {i+1}. {q}")

    # 2. 각 쿼리마다 검색 수행
    all_results = []
    for query in all_queries:
        results = retriever.invoke(query)
        all_results.append(results)

    # 3. RRF로 결과 융합
    fused_docs = reciprocal_rank_fusion(all_results)

    return fused_docs[:top_k]

### 실행

In [ ]:
question = "GPT의 발전 과정과 현재 성능"
fused_results = rag_fusion_retrieve(question, top_k=4)

print(f"\n=== RAG-Fusion 결과 (상위 {len(fused_results)}개) ===")
for i, doc in enumerate(fused_results, 1):
    print(f"\n[{i}] {doc.page_content}")

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


생성된 쿼리 (4개):
  1. GPT의 발전 과정과 현재 성능
  2. GPT 발전 과정 논문
  3. GPT 현재 성능 평가
  4. GPT 미래 전망 분석

=== RAG-Fusion 결과 (상위 4개) ===

[1] GPT 시리즈의 발전 과정을 살펴보면, GPT-3에서 GPT-4로 넘어오면서 논리적 추론과 맥락 이해 능력이 크게 개선되었습니다.

[2] OpenAI가 GPT-5를 발표했습니다. 새로운 모델은 이전 버전보다 추론 능력이 크게 향상되었으며, 멀티모달 기능이 강화되었습니다.

[3] 인공지능 기술의 발전으로 자연어 처리 분야가 빠르게 성장하고 있습니다. 특히 대형 언어 모델(LLM)의 성능 향상이 눈에 띕니다.

[4] 최신 AI 모델들은 코드 생성, 문서 요약, 번역 등 다양한 작업에서 인간 수준의 성능을 보이고 있습니다.


## RAG 체인 + RAG-Fusion
: 다른 문서 사용

### Document Loader

In [ ]:
urls = [
    "https://www.ibm.com/kr-ko/think/topics/agentic-rag",
    "https://www.ibm.com/kr-ko/think/topics/retrieval-augmented-generation",
    "https://www.ibm.com/kr-ko/think/topics/ai-agents",
    "https://www.ibm.com/kr-ko/think/topics/react-agent",
    "https://www.ibm.com/kr-ko/think/topics/agentic-chunking",
]

loader = WebBaseLoader(
    web_paths=urls,
    bs_kwargs=dict(parse_only=bs4.SoupStrainer(["h2", "h3", "p"])),
)
docs = loader.load()
print(len(docs), "페이지 로드")

5 페이지 로드


### Embedding & Chunking

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-m3')

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,              # 앞에서 만든 bge-m3 토크나이저
    chunk_size=500,
    chunk_overlap=100,
)
documents = text_splitter.split_documents(docs)
print('청크 수:', len(documents))

청크 수: 70


In [ ]:
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 10})

### rag 프롬프트

In [ ]:
rag_prompt = ChatPromptTemplate.from_template("""
다음 컨텍스트를 기반으로 질문에 답변하세요.

컨텍스트:
{context}

질문: {question}

답변:
""")

### format_docs 함수
: 검색된 document 리스트에서 본문만 꺼내 \n\n로 이어붙여 하나의 문자열로 만듦

In [ ]:
def format_docs(docs: List[Document]) -> str:
    return "\n\n".join(doc.page_content for doc in docs)

### rag_fusion_chain

In [ ]:
rag_fusion_chain = (
    {
        "context": lambda x: format_docs(rag_fusion_retrieve(x["question"])),
        "question": lambda x: x["question"]
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

### 단일 vs 다중 성능 비교

In [ ]:
# 쿼리마다 더 넓게 검색하도록 3에서 10으로 확대
retriever.search_kwargs['k'] = 10

def compare_retrieval(question: str, top_k: int=5):
    """단일 쿼리와 RAG-Fusion 결과를 비교합니다."""

    # 단일 쿼리 검색
    single_results = retriever.invoke(question)[:top_k]

    # RAG-Fusion 검색
    fusion_results = rag_fusion_retrieve(question, top_k=top_k)

    print(f"질문: {question}\n")

    print("=== 단일 쿼리 결과 ===")
    for i, doc in enumerate(single_results, 1):
        print(f"[{i}] {' '.join(doc.page_content.split())[:60]}...")

    print("\n=== RAG-Fusion 결과 ===")
    for i, doc in enumerate(fusion_results, 1):
        print(f"[{i}] {' '.join(doc.page_content.split())[:60]}...")

    # 차이점 분석
    single_contents = {doc.page_content for doc in single_results}
    fusion_contents = {doc.page_content for doc in fusion_results}

    only_in_fusion = fusion_contents - single_contents
    print(f"\n추가로 찾은 문서: {len(only_in_fusion)}개")

In [ ]:
compare_retrieval("RAG에서 에이전트는 어떤 역할을 하고, ReAct는 거기서 어떻게 쓰여?")

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


생성된 쿼리 (4개):
  1. RAG에서 에이전트는 어떤 역할을 하고, ReAct는 거기서 어떻게 쓰여?
  2. RAG 에이전트 역할
  3. ReAct 알고리즘 구현
  4. RAG과의 비교 연구
질문: RAG에서 에이전트는 어떤 역할을 하고, ReAct는 거기서 어떻게 쓰여?

=== 단일 쿼리 결과 ===
[1] 작성자 Senior Staff Writer, AI ModelsIBM Think ReAct 에이전트란 무엇인가...
[2] 사용하여 다른 AI 모델을 관리하는 프로세스는 AI 오케스트레이션의 한 유형입니다.ReAct 에이전트ReAc...
[3] 에이전틱 RAG는 하나 이상의 AI 에이전트를 RAG 시스템에 통합하는 방식으로 작동합니다. 예를 들어, 에...
[4] 그 새로운 정보를 사용하여 (“날씨가 추울 거야”), 다음 고려 사항(“내게 따뜻한 옷이 무엇이 있을까?”)...
[5] 더 작은, 더 효율적인 모델로 구축된 에이전트들에게 하위 작업을 위임하는 방식으로 구성될 수 있습니다.ReA...

=== RAG-Fusion 결과 ===
[1] 에이전틱 RAG는 하나 이상의 AI 에이전트를 RAG 시스템에 통합하는 방식으로 작동합니다. 예를 들어, 에...
[2] 작성자 Senior Staff Writer, AI ModelsIBM Think ReAct 에이전트란 무엇인가...
[3] 에이전틱 RAG와 기존 RAG 시스템 비교...
[4] 사용하여 다른 AI 모델을 관리하는 프로세스는 AI 오케스트레이션의 한 유형입니다.ReAct 에이전트ReAc...
[5] 그 새로운 정보를 사용하여 (“날씨가 추울 거야”), 다음 고려 사항(“내게 따뜻한 옷이 무엇이 있을까?”)...

추가로 찾은 문서: 1개


**결과해석**

RRF로 질문에 가장 직접적인 문서가 3위 → 1위로 상승

의도에서 벗어난 쿼리가 새 문서를 끌어옴 → RRF는 영향을 줄일 뿐 완전히 막진 못함

쿼리가 한쪽 측면(RAG)에 치우쳐 다른 측면(ReAct) 문서가 밀려남

모든 쿼리에 걸리는 노이즈 청크(작성자 소개)는 오히려 상위 유지 → 리랭커·전처리가 필요